In [4]:
!pip -q install gradio pydantic requests pandas
!pip -q install python-docx PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.8 MB/s eta 0:00:00


In [7]:
import os

# LLM OpenRouter
os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["LLM_API_KEY"]  = "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"
os.environ["LLM_MODEL"]    = "mistralai/mistral-7b-instruct:free"

# Adzuna tab
os.environ["ADZUNA_APP_ID"]  = "e95ec8af"
os.environ["ADZUNA_APP_KEY"] = "e371d7574326120f73de0cecb87eb2c0"


In [8]:
# Recruitment Assistant
import os, json, textwrap, traceback, html, sqlite3, re
from typing import Optional, Dict, Any, List
import requests
import pandas as pd
from pydantic import BaseModel, Field
import gradio as gr

from PyPDF2 import PdfReader
from docx import Document

DB_PATH = "/content/recruit.db"

# DB bootstrap
def _init_db():
    with sqlite3.connect(DB_PATH) as con:
        # leads table
        con.execute("""
        CREATE TABLE IF NOT EXISTS leads (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT, title TEXT, company TEXT, email TEXT, linkedin TEXT,
            industry TEXT, company_size TEXT, role TEXT, location TEXT, notes TEXT,
            created_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        # candidates table
        con.execute("""
        CREATE TABLE IF NOT EXISTS candidates (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT, email TEXT, phone TEXT, location TEXT,
            skills TEXT, availability TEXT, notes TEXT,
            match_score INTEGER, matched_skills TEXT, missing_skills TEXT,
            source_file TEXT, resume_text TEXT,
            created_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
_init_db()

def _df(sql: str, params: tuple = ()):
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql_query(sql, con, params=params)

def _exec(sql: str, params: tuple = ()):
    with sqlite3.connect(DB_PATH) as con:
        con.execute(sql, params); con.commit()

# LLM helpers
class InboxResult(BaseModel):
    label: str = Field(description="Interested | Not Interested | Needs Follow-up")
    confidence: float
    summary: str
    reply_draft: str = ""

def _pyd_dump(model: BaseModel) -> Dict[str, Any]:
    return model.model_dump() if hasattr(model, "model_dump") else model.dict()

def _extract_json_block(text: str) -> str:
    text = text.strip()
    try:
        json.loads(text); return text
    except Exception:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1 and end > start: return text[start:end+1]
        raise ValueError("No JSON object found in model output.")

def llm_chat_json(system_prompt: str, user_prompt: str, schema_hint: str) -> Dict[str, Any]:
    LLM_API_KEY  = os.getenv("LLM_API_KEY")
    LLM_MODEL    = os.getenv("LLM_MODEL", "mistralai/mistral-7b-instruct:free")
    LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://openrouter.ai/api/v1")
    if not LLM_API_KEY: raise RuntimeError("Missing LLM_API_KEY")

    headers = {"Authorization": f"Bearer {LLM_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
            {"role": "system", "content": f"Output ONLY valid JSON that matches this schema. No markdown, no commentary.\nSchema:\n{schema_hint}"}
        ],
        "temperature": 0.2
    }
    r = requests.post(f"{LLM_BASE_URL.rstrip('/')}/chat/completions", headers=headers, json=payload, timeout=90)
    r.raise_for_status()
    text = r.json()["choices"][0]["message"]["content"]
    return json.loads(_extract_json_block(text))

#Existing features
def inbox_brain(email_text: str, tone: str = "neutral") -> Dict[str, Any]:
    system_prompt = "You classify recruiting emails and draft concise replies."
    user_prompt = f"""
Email:
\"\"\"{email_text}\"\"\"

Tasks:
1) Classify as: Interested, Not Interested, or Needs Follow-up.
2) Provide a 1-paragraph summary.
3) Draft a short reply in a {tone} tone.

Return JSON exactly as:
{{
  "inbox": {{
    "label": "Interested|Not Interested|Needs Follow-up",
    "confidence": 0.0-1.0,
    "summary": "string",
    "reply_draft": "string"
  }}
}}
"""
    schema_hint = """
{
  "inbox": {
    "label": "string",
    "confidence": 0.0,
    "summary": "string",
    "reply_draft": "string"
  }
}
"""
    raw = llm_chat_json(system_prompt, user_prompt, schema_hint)
    maybe = raw.get("inbox", raw)
    if isinstance(maybe, dict):
        result = _pyd_dump(InboxResult(**maybe))
    elif isinstance(maybe, list):
        first = maybe[0] if maybe else {"label":"Needs Follow-up","confidence":0.5,"summary":"Empty","reply_draft":""}
        result = _pyd_dump(InboxResult(**first))
    else:
        raise TypeError(f"Unexpected inbox payload type: {type(maybe)}")
    return {"inbox": result}

# scoring each candidate
def compare_jd_resume(jd_text: str, resume_text: str) -> Dict[str, Any]:
    system_prompt = "You are a precise resume tailor for data/AI roles."
    user_prompt = f"""
Job Description:
\"\"\"{jd_text}\"\"\"

Candidate Resume:
\"\"\"{resume_text}\"\"\"

Tasks:
- Compute a fit score (0-100).
- List matched_skills and missing_skills (short phrases).
- List keywords_to_add (ATS-friendly).
- Suggest resume_edits with skills_add (short list) and 2-4 experience_bullets.
- Draft a tailored_summary (80-120 words).

Return JSON exactly as:
{{
  "fit_score": 0,
  "matched_skills": [],
  "missing_skills": [],
  "keywords_to_add": [],
  "resume_edits": {{
    "skills_add": [],
    "experience_bullets": []
  }},
  "tailored_summary": ""
}}
"""
    schema_hint = """
{
  "fit_score": 0,
  "matched_skills": ["string"],
  "missing_skills": ["string"],
  "keywords_to_add": ["string"],
  "resume_edits": { "skills_add": ["string"], "experience_bullets": ["string"] },
  "tailored_summary": "string"
}
"""
    return llm_chat_json(system_prompt, user_prompt, schema_hint)

# Resume parsing
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(?:(?:\+?\d{1,3}[-\s]?)?(?:\(?\d{3}\)?[-\s]?)?\d{3}[-\s]?\d{4})")

def _read_pdf(path: str) -> str:
    try:
        pdf = PdfReader(path)
        return "\n".join([page.extract_text() or "" for page in pdf.pages])
    except Exception:
        return ""

def _read_docx(path: str) -> str:
    try:
        doc = Document(path)
        return "\n".join([p.text for p in doc.paragraphs])
    except Exception:
        return ""

def _read_txt(path: str) -> str:
    try:
        with open(path, "rb") as f:
            return f.read().decode("utf-8", errors="ignore")
    except Exception:
        return ""

def extract_text_from_file(path: str) -> str:
    p = path.lower()
    if p.endswith(".pdf"):  return _read_pdf(path)
    if p.endswith(".docx"): return _read_docx(path)
    if p.endswith(".txt"):  return _read_txt(path)
    # fallback: try text read
    return _read_txt(path)

def quick_profile_guess(resume_text: str, filename: str = "") -> Dict[str,str]:
    email = EMAIL_RE.search(resume_text)
    phone = PHONE_RE.search(resume_text)

    first_line = next((ln.strip() for ln in resume_text.splitlines() if ln.strip()), "")
    first_line = re.sub(r"[^A-Za-z .'-]", " ", first_line).strip()
    guessed_name = first_line[:80] if 2 <= len(first_line.split()) <= 6 else ""
    if not guessed_name and filename:
        base = re.sub(r"\.[^.]+$", "", filename)
        guessed_name = re.sub(r"[_-]", " ", base).title()
    return {
        "name": guessed_name,
        "email": email.group(0) if email else "",
        "phone": phone.group(0) if phone else "",
        "location": ""
    }

def upsert_candidate(rec: Dict[str, Any]) -> None:
    # if email exists, update; else insert
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        email = (rec.get("email") or "").strip().lower()
        cur.execute("SELECT id FROM candidates WHERE lower(email)=? LIMIT 1", (email,))
        row = cur.fetchone()
        fields = ["name","email","phone","location","skills","availability","notes",
                  "match_score","matched_skills","missing_skills","source_file","resume_text"]
        vals = [rec.get(k) for k in fields]
        if row:
            set_clause = ",".join([f"{k}=?" for k in fields])
            cur.execute(f"UPDATE candidates SET {set_clause} WHERE id=?", vals+[row[0]])
        else:
            cur.execute(f"INSERT INTO candidates ({','.join(fields)}) VALUES ({','.join(['?']*len(fields))})", vals)
        con.commit()

def query_candidates(search: str, min_match: int, skills_include: List[str]) -> pd.DataFrame:
    sql = """SELECT id,name,email,phone,location,match_score,matched_skills,missing_skills,availability,notes,source_file,created_ts
             FROM candidates ORDER BY match_score DESC NULLS LAST, created_ts DESC"""
    df = _df(sql)
    if search:
        s = search.lower()
        df = df[df.apply(lambda r: s in str(r.to_dict()).lower(), axis=1)]
    if min_match and int(min_match) > 0:
        df = df[(df["match_score"].fillna(0).astype(int) >= int(min_match))]
    if skills_include:
        want = [w.strip().lower() for w in skills_include if w.strip()]
        if want:
            df = df[df["matched_skills"].fillna("").str.lower().apply(lambda s: all(w in s for w in want))]
    return df

def save_candidate_edits(df: pd.DataFrame):
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor()
        for _, r in df.iterrows():
            if pd.isna(r.get("id")): continue
            cur.execute("""UPDATE candidates
                           SET name=?, email=?, phone=?, location=?, availability=?, notes=?
                           WHERE id=?""",
                        (str(r.get("name") or ""), str(r.get("email") or ""), str(r.get("phone") or ""),
                         str(r.get("location") or ""), str(r.get("availability") or ""), str(r.get("notes") or ""), int(r["id"])))
        con.commit()

def delete_candidate_ids(id_csv: str) -> int:
    ids = [s.strip() for s in (id_csv or "").split(",") if s.strip().isdigit()]
    if not ids: return 0
    with sqlite3.connect(DB_PATH) as con:
        cur = con.cursor(); cur.execute("DELETE FROM candidates WHERE id IN (" + ",".join(["?"]*len(ids)) + ")", [int(x) for x in ids])
        n = cur.rowcount; con.commit()
        return n

# Scoring pipeline
def score_candidate_pipeline(jd_text: str, files: List[Any]) -> Dict[str, Any]:
    if not jd_text.strip():
        raise ValueError("Paste a Job Description first.")
    processed = []
    errors = []
    for f in (files or []):
        path = getattr(f, "name", str(f))
        text = extract_text_from_file(path)
        if not text.strip():
            errors.append(f"Could not parse: {path}")
            continue
        prof = quick_profile_guess(text, filename=os.path.basename(path))
        try:
            comp = compare_jd_resume(jd_text, text)
            match_score = int(comp.get("fit_score", 0))
            matched = ", ".join(comp.get("matched_skills", [])[:30])
            missing = ", ".join(comp.get("missing_skills", [])[:30])
            skills = ", ".join(sorted(set(comp.get("matched_skills", []) + comp.get("resume_edits",{}).get("skills_add",[])))[:40])
        except Exception as e:

            match_score, matched, missing, skills = None, "", "", ""
            errors.append(f"{os.path.basename(path)} → LLM scoring failed: {e}")

        rec = {
            "name": prof["name"],
            "email": prof["email"],
            "phone": prof["phone"],
            "location": prof["location"],
            "skills": skills,
            "availability": "",
            "notes": "",
            "match_score": match_score,
            "matched_skills": matched,
            "missing_skills": missing,
            "source_file": os.path.basename(path),
            "resume_text": text
        }
        upsert_candidate(rec)
        processed.append(rec["email"] or rec["name"] or rec["source_file"])

    table = query_candidates("", 0, [])
    return {
        "processed": processed,
        "errors": errors,
        "table": table
    }

# UI
CSS = """
.wrap pre { white-space: pre-wrap !important; }
.small .gr-textbox textarea { height: 160px !important; }
"""

with gr.Blocks(css=CSS, title="Recruitment Assistant") as demo:
    gr.Markdown("## Recruitment Assistant")

    # Inbox
    with gr.Tab("Inbox: Classify + Draft"):
        with gr.Row():
            email_in = gr.Textbox(label="Paste email text", lines=12, elem_classes=["small"])
            tone_in  = gr.Dropdown(choices=["neutral","friendly","formal"], value="neutral", label="Reply tone")
        run_inbox = gr.Button("Classify + Draft Reply", variant="primary")
        with gr.Row():
            inbox_json = gr.JSON(label="Result (JSON)", elem_classes=["wrap"])
            reply_out  = gr.Textbox(label="Reply Draft (editable)", lines=10)
        err_box = gr.Markdown(visible=False)

        def _inbox_api(email_text, tone):
            try:
                out = inbox_brain(email_text or "", tone or "neutral")
                reply = (out.get("inbox") or {}).get("reply_draft", "")
                return out, reply, gr.update(visible=False, value="")
            except Exception as e:
                trace = traceback.format_exc()
                err_md = f"**Error:** {html.escape(str(e))}\n\n```\n{trace}\n```"
                return {"error": str(e)}, "", gr.update(visible=True, value=err_md)

        run_inbox.click(_inbox_api, inputs=[email_in, tone_in], outputs=[inbox_json, reply_out, err_box])

    # Jobs
    from math import inf
    COUNTRY_OPTS = {
        "US": "us", "UK": "gb", "Canada": "ca", "Australia": "au", "India": "in",
        "Germany": "de", "France": "fr", "Netherlands": "nl", "Ireland": "ie",
        "Spain": "es", "Italy": "it", "Poland": "pl", "Portugal": "pt",
        "Switzerland": "ch", "Austria": "at", "Sweden": "se", "Norway": "no",
        "Denmark": "dk", "Belgium": "be", "Mexico": "mx", "Brazil": "br",
        "New Zealand": "nz", "Singapore": "sg", "South Africa": "za", "UAE": "ae"
    }
    COUNTRY_LIST = list(COUNTRY_OPTS.keys())

    def _looks_remote(texts: List[str]) -> bool:
        check = " ".join([t for t in texts if t]).lower()
        return ("remote" in check) or ("work from home" in check) or ("wfh" in check) or ("hybrid" in check)

    def _company_match(name: str, allow: List[str], deny: List[str]) -> bool:
        n = (name or "").lower()
        if allow and not any(a in n for a in allow): return False
        if deny and any(d in n for d in deny): return False
        return True

    def _salary_overlaps(job_min: Optional[float], job_max: Optional[float], sel_min: int, sel_max: int) -> bool:
        if job_min is None and job_max is None: return True
        lo = job_min if job_min is not None else job_max
        hi = job_max if job_max is not None else job_min
        if lo is None and hi is None: return True
        if lo is None: lo = hi
        if hi is None: hi = lo
        return (hi >= sel_min) and (lo <= sel_max)

    def adzuna_search(query: str, location_text: str, country_label: str, page: int = 1, results_per_page: int = 50) -> Dict[str, Any]:
        ADZUNA_APP_ID  = os.getenv("ADZUNA_APP_ID")
        ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")
        if not ADZUNA_APP_ID or not ADZUNA_APP_KEY: raise RuntimeError("Missing ADZUNA_APP_ID / ADZUNA_APP_KEY")
        country_code = COUNTRY_OPTS.get(country_label, "us")
        url = f"https://api.adzuna.com/v1/api/jobs/{country_code}/search/{max(1,int(page or 1))}"
        params = {
            "app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY,
            "what": (query or "").strip(), "where": (location_text or "").strip(),
            "results_per_page": results_per_page, "content-type": "application/json"
        }
        r = requests.get(url, params=params, timeout=60); r.raise_for_status(); data = r.json()
        rows = []
        for it in data.get("results", []):
            title   = it.get("title") or ""
            company = (it.get("company") or {}).get("display_name") or ""
            loc     = (it.get("location") or {}).get("display_name") or ""
            link    = it.get("redirect_url") or ""
            desc    = textwrap.shorten((it.get("description") or "").replace("\n"," "), width=220)
            rows.append({
                "title": title, "company": company, "location": loc,
                "salary_min": it.get("salary_min"), "salary_max": it.get("salary_max"),
                "created": it.get("created"), "contract_type": it.get("contract_type"),
                "description": desc, "link": link
            })
        return {"table": pd.DataFrame(rows), "raw": data}

    def render_cards(df: pd.DataFrame) -> str:
        if df is None or df.empty: return "No results."
        cards = []
        for _, r in df.iterrows():
            title = html.escape(str(r.get("title",""))); company = html.escape(str(r.get("company","")))
            loc = html.escape(str(r.get("location",""))); desc = html.escape(str(r.get("description","")))
            link = html.escape(str(r.get("link",""))); created = html.escape(str(r.get("created","")))
            salary = []
            if not pd.isna(r.get("salary_min")): salary.append(f"${int(r['salary_min']):,}")
            if not pd.isna(r.get("salary_max")): salary.append(f"${int(r['salary_max']):,}")
            salary_str = " – ".join(salary) if salary else "—"
            cards.append(f"""
<div style="border:1px solid #eee;border-radius:14px;padding:14px;margin:10px 0;box-shadow:0 1px 4px rgba(0,0,0,0.04)">
  <div style="font-weight:600;font-size:16px;margin-bottom:6px">{title}</div>
  <div style="color:#555;margin-bottom:8px">{company} • {loc}</div>
  <div style="color:#333;margin-bottom:8px">{desc}</div>
  <div style="font-size:12px;color:#666;margin-bottom:10px">Salary: {salary_str} &nbsp;|&nbsp; Posted: {created}</div>
  <a href="{link}" target="_blank" style="display:inline-block;background:#2563eb;color:#fff;padding:8px 12px;border-radius:10px;text-decoration:none">Apply</a>
</div>""")
        return "\n".join(cards)

    with gr.Tab("Jobs: Global Adzuna Search"):
        with gr.Row():
            q_in   = gr.Textbox(label="Role / keywords", value="data analyst")
            loc_in = gr.Textbox(label="City/Region (optional)", value="")
            country_in = gr.Dropdown(choices=COUNTRY_LIST, value="US", label="Country")
        with gr.Row():
            sal_min = gr.Slider(label="Min salary (USD)", minimum=0, maximum=300000, step=1000, value=0)
            sal_max = gr.Slider(label="Max salary (USD)", minimum=0, maximum=300000, step=1000, value=300000)
            remote_only = gr.Checkbox(label="Remote / WFH / Hybrid only (best-effort)", value=False)
        with gr.Row():
            include_comp = gr.Textbox(label="Company include (comma-separated)")
            exclude_comp = gr.Textbox(label="Company exclude (comma-separated)")
            posted_days  = gr.Slider(label="Posted within (days)", minimum=0, maximum=60, step=1, value=0)
        with gr.Row():
            page_in= gr.Number(label="Page", value=1, precision=0)
            show_cards = gr.Checkbox(label="Show card view", value=True)

        do_search = gr.Button("Search Jobs", variant="primary")
        jobs_table = gr.Dataframe(label="Results (sortable)", wrap=True, interactive=False, visible=True)
        links_md   = gr.HTML(label="Card View", value="")
        jobs_err   = gr.Markdown(visible=False)

        def _job_api(q, loc, country_label, smin, smax, rem_only, inc_c, exc_c, days, page, cards):
            try:
                res = adzuna_search(q or "", loc or "", country_label, int(page or 1), results_per_page=50)
                df = res["table"]
                inc = [x.strip().lower() for x in (inc_c or "").split(",") if x.strip()]
                exc = [x.strip().lower() for x in (exc_c or "").split(",") if x.strip()]
                if rem_only:
                    df = df[df.apply(lambda r: _looks_remote([r.get("title"), r.get("location"), r.get("description")]), axis=1)]
                if inc or exc:
                    df = df[df.apply(lambda r: _company_match(str(r.get("company","")), inc, exc), axis=1)]
                df = df[df.apply(lambda r: _salary_overlaps(
                    None if pd.isna(r.get("salary_min")) else float(r.get("salary_min")),
                    None if pd.isna(r.get("salary_max")) else float(r.get("salary_max")),
                    int(smin), int(smax)), axis=1)]
                if days and int(days) > 0:
                    ts_now = pd.Timestamp.utcnow()
                    df["created_ts"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
                    cutoff = ts_now - pd.Timedelta(days=int(days))
                    df = df[df["created_ts"] >= cutoff].drop(columns=["created_ts"])
                if df is None or df.empty:
                    df_display = pd.DataFrame(columns=["title","company","location","salary_min","salary_max","created","contract_type","description","link"])
                    cards_html = "No results."
                else:
                    df_display = df.copy()
                    cards_html = render_cards(df) if cards else ""
                return df_display, cards_html, gr.update(visible=False, value="")
            except Exception as e:
                trace = traceback.format_exc()
                err_md = f"**Job search error:** {html.escape(str(e))}\n\n```\n{trace}\n```"
                return pd.DataFrame(), "", gr.update(visible=True, value=err_md)
        do_search.click(_job_api,
                        inputs=[q_in, loc_in, country_in, sal_min, sal_max, remote_only, include_comp, exclude_comp, posted_days, page_in, show_cards],
                        outputs=[jobs_table, links_md, jobs_err])

    # JD single compare
    with gr.Tab("JD ↔ Resume Compare"):
        with gr.Row():
            jd_in = gr.Textbox(label="Job Description (paste)", lines=14)
            resume_in = gr.Textbox(label="Your Resume (paste)", lines=14)
        run_compare = gr.Button("Analyze Fit", variant="primary")
        result_json = gr.JSON(label="Raw Result (JSON)")
        summary_out = gr.Textbox(label="Tailored Summary (editable)", lines=6)
        advice_md   = gr.Markdown(label="What to change/add")
        def _compare_api(jd, resume):
            try:
                out = compare_jd_resume(jd or "", resume or "")
                ms = out.get("missing_skills", []); kw = out.get("keywords_to_add", [])
                edits = out.get("resume_edits", {}) or {}
                bullets = edits.get("experience_bullets", []); skills_add = edits.get("skills_add", [])
                fit = out.get("fit_score", 0)
                md = [f"**Fit score:** {fit}/100"]
                if ms: md.append("**Missing skills:** " + ", ".join(ms))
                if kw: md.append("**Keywords to add (ATS):** " + ", ".join(kw))
                if skills_add: md.append("**Add to Skills section:** " + ", ".join(skills_add))
                if bullets:
                    md.append("**New experience bullets:**")
                    md.extend([f"- {b}" for b in bullets])
                return out, out.get("tailored_summary",""), "\n\n".join(md) or "No gaps found."
            except Exception as e:
                trace = traceback.format_exc()
                return {"error": str(e)}, "", f"**Compare error:** {html.escape(str(e))}\n\n```\n{trace}\n```"
        run_compare.click(_compare_api, inputs=[jd_in, resume_in], outputs=[result_json, summary_out, advice_md])

    # Candidates
    with gr.Tab("Candidates: Upload, Parse, Match"):
        gr.Markdown("**Step 1:** Paste the Job Description, then upload resumes (PDF/DOCX/TXT).")
        jd_cand = gr.Textbox(label="Job Description for matching", lines=10)
        files_up = gr.File(label="Upload resumes", file_count="multiple", file_types=[".pdf",".docx",".txt"])
        run_score = gr.Button("Parse & Score", variant="primary")

        status_md = gr.Markdown(visible=False)

        gr.Markdown("**Filters**")
        with gr.Row():
            search_txt = gr.Textbox(label="Search (name/email/notes/skills/etc.)")
            min_match  = gr.Slider(label="Min match %", minimum=0, maximum=100, value=0, step=1)
            skills_inc = gr.Textbox(label="Skills include (comma-separated)", placeholder="python, sql, pandas")

        cand_tbl = gr.Dataframe(label="Candidates (editable: name/email/phone/location/availability/notes)", interactive=True, wrap=True)
        save_cand = gr.Button("Save edits")
        del_ids   = gr.Textbox(label="Delete by ID(s), comma-separated")
        del_btn   = gr.Button("Delete")
        export_btn= gr.Button("Export filtered to CSV")
        export_file = gr.File(label="Download CSV")

        def _do_score(jd, files):
            try:
                out = score_candidate_pipeline(jd or "", files or [])
                msg = "✅ Processed: " + ", ".join(out["processed"]) if out["processed"] else "No files processed."
                if out["errors"]:
                    msg += "<br>" + "<br>".join([html.escape(e) for e in out["errors"]])
                table = query_candidates("", 0, [])
                return gr.update(visible=True, value=msg), table
            except Exception as e:
                trace = traceback.format_exc()
                return gr.update(visible=True, value=f"**Error:** {html.escape(str(e))}\n\n```\n{trace}\n```"), query_candidates("", 0, [])

        def _refresh_table(s, m, skills):
            skills_list = [x.strip() for x in (skills or "").split(",") if x.strip()]
            return query_candidates(s or "", int(m or 0), skills_list)

        def _save(df):
            try:
                save_candidate_edits(df if isinstance(df, pd.DataFrame) else pd.DataFrame(df))
                return gr.update(visible=True, value="✅ Saved edits")
            except Exception as e:
                return gr.update(visible=True, value=f"**Save error:** {html.escape(str(e))}")

        def _delete(ids):
            n = delete_candidate_ids(ids)
            return gr.update(visible=True, value=f"🗑️ Deleted {n} row(s)")

        def _export(s, m, skills):
            df = _refresh_table(s, m, skills)
            path = "/content/candidates_export.csv"
            df.to_csv(path, index=False)
            return path

        run_score.click(_do_score, inputs=[jd_cand, files_up], outputs=[status_md, cand_tbl])
        for comp in (search_txt, min_match, skills_inc):
            comp.change(_refresh_table, inputs=[search_txt, min_match, skills_inc], outputs=[cand_tbl])
        save_cand.click(_save, inputs=[cand_tbl], outputs=[status_md])
        del_btn.click(_delete, inputs=[del_ids], outputs=[status_md])
        export_btn.click(_export, inputs=[search_txt, min_match, skills_inc], outputs=[export_file])

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7905e6d9cf980732e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7905e6d9cf980732e3.gradio.live
